In [1]:
import pandas as pd
import re
# Load ratings data
ratings_data = pd.read_csv('ml-latest-small/ratings.csv')

# Load movies data
movies_data = pd.read_csv('ml-latest-small/movies.csv')


In [2]:
# Merge ratings data with movies data
data = pd.merge(ratings_data, movies_data, on='movieId')
# Extract genres from title
data['genres'] = data['genres'].str.split('|')
genres = data['genres'].explode().unique()
for genre in genres:
    data[genre] = data['genres'].apply(lambda x: 1 if genre in x else 0)


In [3]:
# Extract release year from title
data['year'] = data['title'].str.extract('\((\d{4})\)')


In [4]:
from sklearn.model_selection import train_test_split

# Split data into training and testing sets
X = data.drop(['userId', 'movieId', 'title', 'genres', 'rating', 'timestamp', 'year'], axis=1)
y = data['rating']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [5]:
from sklearn.linear_model import LinearRegression

# Train linear regression model
reg = LinearRegression()
reg.fit(X_train, y_train)


LinearRegression()

In [6]:
from sklearn.metrics import mean_squared_error

# Test linear regression model
y_pred = reg.predict(X_test)
rmse = mean_squared_error(y_test, y_pred, squared=False)
print(f'RMSE: {rmse:.2f}')


RMSE: 1.02


In [7]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from deap import base, creator, tools, algorithms
from sklearn.metrics import mean_squared_error
import numpy as np
import random
import operator
import warnings

warnings.filterwarnings('ignore')

def evaluate(params):
    # Create pipeline
    pipeline = make_pipeline(StandardScaler(), LinearRegression(**params))
    
    # Calculate cross-validation score
    scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_squared_error')
    
    # Calculate fitness (MSE)
    mse = -np.mean(scores)
    
    return mse,


In [8]:
# Define genetic algorithm parameters
population_size = 50
num_generations = 10
mutation_rate = 0.2
crossover_rate = 0.5
num_params = len(X_train.columns)
param_range = {'normalize': [True, False]}


In [9]:

## Data structure and initializer creation
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)
toolbox = base.Toolbox()
toolbox.register("attr_float", random.random)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_float, 10)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def onemax(individual):
    return sum(individual),

toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=1, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)
toolbox.register("evaluate", onemax)


In [10]:
# Run the genetic algorithm
pop = toolbox.population(n=population_size)
# Crossover probability
CXPB = 0.5
# Mutation probability
MUTPB = 0.2
param_values = []
for generation in range(num_generations):
    print(f'Generation {generation+1}...')
    offspring = algorithms.varAnd(pop, toolbox, cxpb=CXPB, mutpb=MUTPB)
    fits = toolbox.map(toolbox.evaluate, offspring)
    for fit, ind in zip(fits, offspring):
        ind.fitness.values = fit
        pop = toolbox.select(offspring + pop, k=population_size)

# Get the best individual
best_param = tools.selBest(pop, k=1)[0]
# Create a pipeline with the best hyperparameters
best_pipeline = make_pipeline(StandardScaler(), LinearRegression(*best_param))
best_pipeline.fit(X_train, y_train)
# Evaluate the pipeline on the test set
y_pred = best_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print(f'Test MSE: {mse:.4f}')

Generation 1...
Generation 2...
Generation 3...
Generation 4...
Generation 5...
Generation 6...
Generation 7...
Generation 8...
Generation 9...
Generation 10...
Test MSE: 1.0414
